In [2]:
import os
import re
import glob
import numpy as np
import mne
import wfdb
from tqdm import tqdm
import warnings
from typing import List, Tuple

# =========================================================
# 1) CHANNEL CONFIG
# =========================================================
CH_LABELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ'
]

def normalize_channel_name(ch: str) -> str:
    ch = ch.upper().strip()
    ch = ch.replace("EEG ", "")
    ch = ch.replace(" ", "")
    parts = ch.split("-")
    if len(parts) > 2:
        ch = "-".join(parts[:2])
    return ch


def pick_required_channels(raw, wanted_channels):
    raw_map = {}

    for ch in raw.ch_names:
        norm = normalize_channel_name(ch)
        raw_map.setdefault(norm, []).append(ch)

    picks = []
    missing = []

    for ch in wanted_channels:
        norm = normalize_channel_name(ch)
        if norm in raw_map:
            picks.append(raw_map[norm][0])  # lấy channel đầu tiên
        else:
            missing.append(ch)

    if missing:
        print("AVAILABLE:", raw_map.keys())
        raise ValueError(f"Missing channels: {missing}")

    raw.pick(picks)
    return raw


# =========================================================
# 2) READ ANNOTATION
# =========================================================
def read_seizures_sec(edf_path: str) -> List[Tuple[float, float]]:
    folder = os.path.dirname(edf_path)
    record_name = os.path.splitext(os.path.basename(edf_path))[0]
    wfdb_record = os.path.join(folder, record_name)

    try:
        ann = wfdb.rdann(wfdb_record, "edf.seizures")
    except Exception:
        return []

    fs = float(ann.fs)
    seizures = []

    for i in range(0, len(ann.sample)-1, 2):
        seizures.append((ann.sample[i]/fs, ann.sample[i+1]/fs))

    return seizures


# =========================================================
# 3) PREPROCESS
# =========================================================
# =========================================================
# 3) PREPROCESS
# =========================================================
def preprocess_raw(raw, config):
    raw = pick_required_channels(raw, CH_LABELS)

    if config['notch_freq']:
        raw.notch_filter(
            np.arange(config['notch_freq'], raw.info['sfreq']/2, config['notch_freq']),
            verbose=False
        )

    raw.filter(config['l_freq'], config['h_freq'], verbose=False)

    if config['resample_sfreq']:
        raw.resample(config['resample_sfreq'], verbose=False)

    # Chuẩn hóa Z-score (Z-score normalization)
    data = raw.get_data()  # Lấy dữ liệu tín hiệu
    # Tính Z-score cho mỗi kênh (mỗi kênh được chuẩn hóa riêng biệt)
    data = (data - np.mean(data, axis=1, keepdims=True)) / np.std(data, axis=1, keepdims=True)
    raw._data = data  # Cập nhật dữ liệu đã chuẩn hóa vào đối tượng raw

    return raw


# =========================================================
# 4) LABELING (GLOBAL STRICT)
# =========================================================
def overlaps(a, b, c, d):
    return (a < d) and (c < b)


def label_window(t_start, t_end, seizures, config):
    sph = config['sph_min'] * 60
    sop = config['sop_min'] * 60
    gap = config['interictal_gap_min'] * 60
    post = config['postictal_min'] * 60

    for i, (s, e) in enumerate(seizures):

        pre_start = s - (sph + sop)
        pre_end   = s - sph

        # ictal
        if overlaps(t_start, t_end, s, e):
            return -1, -1

        # postictal
        if overlaps(t_start, t_end, e, e + post):
            return -1, -1

        # gap BEFORE seizure
        if overlaps(t_start, t_end, s - gap, pre_start):
            return -1, -1

        # gap AFTER seizure 
        if overlaps(t_start, t_end, e, e + gap):
            return -1, -1

        # preictal
        if overlaps(t_start, t_end, pre_start, pre_end):
            if t_start >= pre_start and t_end <= pre_end:
                return 1, i
            else:
                return -1, -1

    return 0, -1


# =========================================================
# 5) GET FILE TIME
# =========================================================
def get_file_start_time(raw):
    meas_date = raw.info.get("meas_date")
    if meas_date is None:
        return None
    return meas_date.timestamp()


# =========================================================
# 6) MAIN PROCESS (GLOBAL TIMELINE)
# =========================================================
def process_patient(input_root, output_root, config):

    files = glob.glob(os.path.join(input_root, "*.edf"))

    file_infos = []

    # ---- lấy thông tin file ----
    for f in files:
        raw = mne.io.read_raw_edf(f, preload=False, verbose=False)
        start = get_file_start_time(raw)
        duration = raw.n_times / raw.info['sfreq']
        file_infos.append((f, start, duration))

    file_infos.sort(key=lambda x: x[1])

    # ---- build global seizure timeline ----
    current_offset = 0
    global_seizures = []

    for f, start, dur in file_infos:
        seizures = read_seizures_sec(f)
        global_seizures += [(s + current_offset, e + current_offset) for s, e in seizures]
        current_offset += dur

    # ---- extract windows ----
    current_offset = 0

    X, y, sid = [], [], []
    starts, ends = [], []

    for f, start, dur in tqdm(file_infos):

        raw = mne.io.read_raw_edf(f, preload=True, verbose=False)

        try:
            raw = preprocess_raw(raw, config)
        except ValueError as e:
            print(f"[SKIP] {f}: {e}")
            current_offset += dur
            continue

        data = raw.get_data()
        sfreq = config['resample_sfreq']

        win = int(config['window_size_sec'] * sfreq)
        step_pre = int(config['preictal_step_sec'] * sfreq)
        step_int = int(config['interictal_step_sec'] * sfreq)

        s = 0
        while s < data.shape[1] - win:

            t_start = current_offset + s / sfreq
            t_end   = current_offset + (s + win) / sfreq

            label, sid_local = label_window(t_start, t_end, global_seizures, config)

            if label == -1:
                s += step_int
                continue

            X.append(data[:, s:s+win])
            y.append(label)
            sid.append(sid_local if sid_local != -1 else -1)

            starts.append(t_start)
            ends.append(t_end)

            s += step_pre if label == 1 else step_int

        current_offset += dur

    # ---- save ----
    X = np.array(X)
    y = np.array(y)
    sid = np.array(sid)

    os.makedirs(output_root, exist_ok=True)

    patient_id = os.path.basename(os.path.normpath(input_root))

    np.savez(
        os.path.join(output_root, f"{patient_id}.npz"),
        X=X,
        y=y,
        seizure_ids=sid,
        sfreq=config['resample_sfreq'],
        starts=np.array(starts),
        ends=np.array(ends)
    )

    print("Saved:", X.shape, "Preictal:", np.sum(y==1))


# =========================================================
# 7) CONFIG
# =========================================================
CONFIG = {
    'window_size_sec': 8.0,
    'preictal_step_sec': 4.0,
    'interictal_step_sec': 4.0,
    'sph_min': 0.0,
    'sop_min': 30.0, # preictal
    'postictal_min': 60.0,
    'interictal_gap_min': 30.0,
    'l_freq': 0.5,
    'h_freq': 40.0,
    'notch_freq': 60.0,
    'resample_sfreq': 256.0
}

# =========================================================
# RUN
# =========================================================
process_patient(
    input_root=r"C:\Users\Admin\Documents\Thesis\CHB-MIT\chb13",
    output_root=r"C:\Users\Admin\Documents\Thesis\Patient-Specific\chb13",
    config=CONFIG
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_8436\761827997.py:173: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8', '-'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(f, preload=False, verbose=False)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8436\761827997.py:173: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(f, preload=False, verbose=False)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8436\761827997.py:173: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8', '-'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(f, preload=False, verbose=False)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8436\761827997.py:173: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(f, preload=False, verbose=False)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8436\761

Saved: (21206, 18, 2048) Preictal: 2044
